# Static discrete choice — pre-code for the class

The skeleton we fill in during the *Programming practice: static discrete choice* class
(Thursday 10 September). The chapter it follows is <https://dse.iskh.me/logit>.

- Work in this notebook as we go; the completed version lands in this folder afterwards
- Add cells as needed, and finish at home whatever is left over
- Nothing here is collected — this is a practical, not a homework

**The model.** A decision maker in state $x \in \{1,2\}$ chooses one of $|D|$ alternatives,
each described by a vector $Y^d$ of $K$ attributes. Two utility specifications:

1. linear, $u(d,x) = Y^d \beta^x$
2. log, $u(d,x) = \ln(Y^d)\,\beta^x$

with $\beta^1 = \beta$ and $\beta^2 = \beta/2$, so the parameters are $\beta \in \mathbb{R}^K$
and the scale $\sigma$. Choice probabilities are logit with scale $\sigma$.

## Part I — numerically safe logit

### 1. The three functions

Write them with **de-maxing**: subtract the maximum along the choice axis before
exponentiating, and divide by $\sigma$ *before* de-maxing. `loglogit` must never take the
log of a probability.

In [ ]:
import numpy as np

def logit(v, sigma=1.0, axis=0):
    """Logit choice probabilities, de-maxed.
    Alternatives along `axis`, any remaining dimensions are states.
    """
    # Your code here

def logsum(v, sigma=1.0, axis=0):
    """Log-sum-exp: expected maximum of v + EV1 shocks, up to sigma*gamma. De-maxed."""
    # Your code here

def loglogit(v, sigma=1.0, axis=0):
    """Log of the logit choice probabilities, computed without logging a probability."""
    # Your code here

### 2. Test them, and break the naive version

Check your three functions against `scipy.special.softmax`, `logsumexp` and `log_softmax`
on random inputs, in both the 1-d and the 2-d (alternatives x states) case.

Then find, by experiment, the utility level at which the naive formula below fails, and
compare it with `np.log(np.finfo(float).max)`. What exactly does it return, and why is
that worse than an exception?

In [ ]:
from scipy.special import softmax, logsumexp, log_softmax

def logit_naive(v, sigma=1.0, axis=0):
    """Logit choice probabilities, straight from the formula — the thing not to do"""
    e = np.exp(np.asarray(v)/sigma)
    return e/e.sum(axis=axis, keepdims=True)

# Your code here

### 3. The logsum is an expected maximum

Draw EV1$(\mu_i,\sigma)$ variables with `rng.gumbel`, take the maximum over alternatives,
and check that its sample mean matches $\sigma\log\sum_i\exp(\mu_i/\sigma) + \sigma\gamma$
(`np.euler_gamma`) and its variance $\sigma^2\pi^2/6$. Use your own `logsum`.

In [ ]:
# Your code here

### 4. Smoothing a kink

For $h(x) = \max(f(x), g(x))$ the smoothed version is
$\tilde h(x) = \sigma\log\big(e^{f(x)/\sigma} + e^{g(x)/\sigma}\big)$, with a known uniform
error $\sigma\ln 2$.

Plot $f$, $g$ and $\tilde h$ for a range of $\sigma$ **using your de-maxed `logsum`**, and
push $\sigma$ down to $10^{-6}$ — where the naive expression cannot go. Verify the
$\sigma\ln 2$ bound numerically.

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(-2, 4, 1000)
f = 2*np.exp(-0.5*(x - 0.5)**2) + 0.3*x**2 - 1
g = -0.4*x**3 + 1.2*x**2 + 0.5*x + 0.5 + 0.3*np.sin(2*x) - 0.5

# Your code here

## Part II — the model in code

The architecture is the one every structural project in this course uses: a model object
holding the parameters and the model parts, a solver, a simulator, a graphical module,
and (later in the semester) an estimator and a counterfactual simulator. Keep these
separate from the *run scripts* that call them.

### 5. The model object

Fill in the class. Two details are not decoration: `__str__` so that printing a model
tells you what it is, and `__setattr__` so that an inconsistent model cannot be created
in the first place.

In [ ]:
class model:
    """Static random utility model with EV1 taste shocks"""

    def __init__(self,
                 label='noname',
                 nalt=5,                    # number of alternatives
                 nattr=3,                   # number of attributes of each alternative
                 attr=None,                 # attribute values Y, (nalt, nattr)
                 param=None,                # structural parameters beta, (nattr,)
                 st=np.array([1, 2]),       # states of the decision maker
                 util_type='linear',        # 'linear' or 'log'
                 sigma=1.0):                # scale of the taste shocks
        """Create a model with the given parameters"""
        # Your code here
        # mind the order of assignments: __setattr__ checks shapes against nalt/nattr

    def __str__(self):
        """Human readable representation of the model"""
        # Your code here

    def __setattr__(self, name, value):
        """Check consistency of the model attributes on assignment"""
        # check shapes of attr and param, and that sigma > 0, then
        super().__setattr__(name, value)

    # ---- model parts ----

    def beta_coef(self):
        """State-specific coefficients, (nattr, nstates)"""
        # Your code here

    def utility(self):
        """Deterministic component of utility, (nalt, nstates)"""
        # Your code here

    def save_solution(self, chpr):
        """Store the computed choice probabilities in the model object"""
        # Your code here

### 6. The solver

Three lines, using your de-maxed `logit`. Assert that the probabilities sum to one — an
axis mistake then shows up immediately instead of inside a likelihood next month.

In [ ]:
def model_solve(m):
    """Solve the model: choice probabilities, alternatives in rows, states in columns"""
    # Your code here

def model_logsolve(m):
    """Log choice probabilities, for the likelihood later in the course"""
    # Your code here

### 7. The simulator

Simulate `nobs` decision makers: draw a state for each, then draw a choice from the
choice probabilities of that state by the inverse-CDF method — `np.cumsum` over
alternatives, a uniform draw, and `np.searchsorted`. Return a `pandas` DataFrame with
`id`, `st`, `choice` and the attributes of the chosen alternative.

Take the random generator as an argument so that a run can be reproduced.

In [ ]:
import pandas as pd

def model_simulate(m, chpr=None, nobs=10, rng=None):
    """Simulate nobs decision makers from a solved model, return a pandas DataFrame"""
    # Your code here

### 8. The graphical module and the dashboard

Each plotting function either makes its own figure, or draws into axes handed to it —
that second option is what lets `plot_dashboard` reuse them without duplicating a line.

Keep a common vertical scale across panels that are meant to be compared.

In [ ]:
def plot_attributes(m, ax=None):
    """Bar plot of the attributes of each alternative"""
    # Your code here

def plot_choice_probabilities(m, ax=None):
    """Bar plot of the model solution, one panel per state"""
    # Your code here

def plot_data(m, data, ax=None):
    """Histograms of simulated choices: pooled, then one panel per state"""
    # Your code here

def plot_dashboard(m, data):
    """Attributes and solution on top, simulated data below"""
    # Your code here

### 9. Run script

Create a model with 5 alternatives, 3 attributes and random attribute values, solve it,
simulate 2500 decision makers, and show the dashboard.

In [ ]:
rng = np.random.default_rng(2026)

# Your code here

## Part III — understanding the model

Answer these with the dashboard, not with algebra. Add code and text cells as needed.

### 10. What does each $\beta_k$ do?

Vary one coefficient at a time. Why does the same change act differently in the two
states?

In [ ]:
# Your code here

### 11. What does $\sigma$ do?

Can you tell $\beta$ apart from $\sigma$ using data on choices alone? Give the argument in
one sentence.

In [ ]:
# Your code here

### 12. Degenerate choice

Can you find parameters that concentrate all simulated choices on one alternative? In how
many different ways? Does your `logit` survive the extreme cases — and would the naive one?

In [ ]:
# Your code here

### 13. Predicted versus simulated behavior

Compare the choice probabilities with histograms of simulated choices for `nobs` = 100 and
100,000. What is the object that estimation recovers, and what does the gap you see here
become in an estimated model?

In [ ]:
# Your code here

### 14. Mimicking a state, and the log specification

Can you mimic the behavior of a decision maker in state $x=1$ using state $x=2$ and a
different $\beta$? What does that tell you about identification?

Do your answers to 10–13 change under the log utility specification?

In [ ]:
# Your code here